# 05 直接MS/MS谱图数据集与简单基线

本步骤只使用冻结的20 ppm Development谱图和结构簇fold，不读取ChemVista标签或性能。目标是在训练复杂Spectrum Transformer之前，验证碎片峰是否提供超出前体与采集元数据的PBT相关性质信息。

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pickle
import platform
import sys
import warnings

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import spearmanr
from sklearn import __version__ as sklearn_version
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, matthews_corrcoef, mean_absolute_error,
    mean_squared_error, r2_score, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

SEED = 42
N_OUTER_FOLDS = 5
TRAIN_SAMPLING_REPEATS = 5
N_LABEL_PERMUTATIONS = 20
N_BOOTSTRAP = 1000
BIN_WIDTH_DA = 1.0
MAX_MZ = 2000.0
N_BINS = int(MAX_MZ / BIN_WIDTH_DA)
MAIN_SPECTRUM_ROUTE = 'fragment_neutral_loss_metadata'
ROUTES = [
    'precursor_mz_only',
    'metadata_only',
    'fragment_mz_presence',
    'fragment_intensity',
    MAIN_SPECTRUM_ROUTE,
]

ROOT = Path.cwd()
if not (ROOT / 'outputs' / 'pbt_msms_v1').exists():
    ROOT = next((p for p in [ROOT, *ROOT.parents] if (p / 'outputs' / 'pbt_msms_v1').exists()), ROOT)
SPLIT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'split_v1'
AUDIT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'spectrum_audit_v1'
STRUCTURE_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'structure_baselines_v1'
OUTPUT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'simple_spectrum_baselines_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'development': SPLIT_DIR / 'development_compounds.csv',
    'split_freeze': SPLIT_DIR / 'split_freeze_manifest.json',
    'split_qc': SPLIT_DIR / 'leakage_checks.csv',
    'public_manifest': AUDIT_DIR / 'public_relevant_spectrum_manifest.pkl',
    'spectrum_audit': AUDIT_DIR / 'spectrum_audit_manifest.json',
    'spectrum_qc': AUDIT_DIR / 'spectrum_audit_qc_checks.csv',
    'public_stream': ROOT / '质谱数据' / 'EnvMolBench_Cleaned.pkl',
    'structure_best': STRUCTURE_DIR / 'structure_baseline_best_models.csv',
    'structure_run_manifest': STRUCTURE_DIR / 'structure_baseline_run_manifest.json',
}
missing = [str(path) for path in PATHS.values() if not path.exists()]
assert not missing, '缺少输入文件：\n' + '\n'.join(missing)

TASKS = {
    'P_screen_not_ready_binary': {'column': 'label__P_screen_not_ready_binary', 'kind': 'classification'},
    'B_LOGBCF_REGRESSION': {'column': 'label__B_LOGBCF_REGRESSION', 'kind': 'regression'},
    'T_EC50_MG_L_REGRESSION': {'column': 'label__T_EC50_MG_L_REGRESSION', 'kind': 'regression'},
    'T_LC50_MG_L_REGRESSION': {'column': 'label__T_LC50_MG_L_REGRESSION', 'kind': 'regression'},
    'T_NOEC_MG_L_REGRESSION': {'column': 'label__T_NOEC_MG_L_REGRESSION', 'kind': 'regression'},
}

print('ROOT:', ROOT)
print('OUTPUT:', OUTPUT_DIR)
print('Python:', sys.executable)
print('scikit-learn:', sklearn_version)

## 1. 读取冻结Development、谱图清单和上游质控

In [ ]:
development = pd.read_csv(PATHS['development'], low_memory=False)
split_qc = pd.read_csv(PATHS['split_qc'])
spectrum_qc = pd.read_csv(PATHS['spectrum_qc'])
public_manifest = pd.read_pickle(PATHS['public_manifest'])
structure_best = pd.read_csv(PATHS['structure_best'])
with PATHS['split_freeze'].open('r', encoding='utf-8') as handle:
    split_freeze = json.load(handle)
with PATHS['spectrum_audit'].open('r', encoding='utf-8') as handle:
    spectrum_audit = json.load(handle)
with PATHS['structure_run_manifest'].open('r', encoding='utf-8') as handle:
    structure_run_manifest = json.load(handle)

spectra = public_manifest.loc[public_manifest['primary_train_eligible_20ppm']].copy()
spectra = spectra.sort_values(['stream_offset', 'spectrum_id']).reset_index(drop=True)
spectra['feature_row'] = np.arange(len(spectra), dtype=int)
eligible_keys = set(spectra['inchikey_2d'])
modeling = development.loc[development['inchikey_2d'].isin(eligible_keys)].copy()
modeling['cv_fold'] = modeling['cv_fold'].astype(int)
modeling = modeling.sort_values('inchikey_2d').reset_index(drop=True)

task_count_rows = []
for task, spec in TASKS.items():
    values = pd.to_numeric(modeling[spec['column']], errors='coerce')
    row = {'task': task, 'kind': spec['kind'], 'n_compounds': int(values.notna().sum())}
    if spec['kind'] == 'classification':
        row.update({'positive': int(values.eq(1).sum()), 'negative': int(values.eq(0).sum()), 'positive_rate': float(values.mean())})
    task_count_rows.append(row)
task_counts = pd.DataFrame(task_count_rows)

print('20 ppm谱图:', len(spectra))
print('20 ppm Development化合物:', len(modeling))
display(task_counts)

## 2. 从未知物可获得信息生成稀疏分箱特征

特征只使用母离子、极性、碰撞能、峰数和清洗后的碎片峰。SMILES和性质标签不参与特征构建。碎片强度使用平方根变换并在每张谱图内归一化；neutral loss仅保留0–2000 Da。

In [ ]:
def binned_max(masses, intensities, *, presence=False):
    masses = np.asarray(masses, dtype=float)
    intensities = np.asarray(intensities, dtype=float)
    valid = np.isfinite(masses) & np.isfinite(intensities) & (masses >= 0) & (masses < MAX_MZ) & (intensities > 0)
    masses, intensities = masses[valid], intensities[valid]
    result = {}
    for bin_index, intensity in zip(np.floor(masses / BIN_WIDTH_DA).astype(int), intensities):
        value = 1.0 if presence else float(intensity)
        result[bin_index] = max(result.get(bin_index, 0.0), value)
    return result

def record_peaks(record):
    tensor = np.asarray(record['tensor'])
    mask = ~np.asarray(record['padding_mask'], dtype=bool)
    mask[0] = False
    mz = tensor[mask, 0].astype(float) * MAX_MZ
    intensity = np.clip(tensor[mask, 3].astype(float), 0, None)
    if len(intensity) and intensity.max() > 0:
        intensity = np.sqrt(intensity / intensity.max())
    valid = np.isfinite(mz) & np.isfinite(intensity) & (mz >= 15.0) & (mz < MAX_MZ) & (intensity > 0)
    return mz[valid], intensity[valid]

fragment_rows, fragment_cols, fragment_data = [], [], []
neutral_rows, neutral_cols, neutral_data = [], [], []
metadata = np.zeros((len(spectra), 6), dtype=np.float32)
order_invariance_pass = True

with PATHS['public_stream'].open('rb') as handle:
    for row_index, row in tqdm(spectra.iterrows(), total=len(spectra), desc='生成1 Da稀疏谱图特征', mininterval=2.0):
        handle.seek(int(row['stream_offset']))
        actual_key, record = pickle.load(handle)
        if actual_key != row['inchikey_2d']:
            raise ValueError(f'谱图索引错位：{row.spectrum_id}')
        mz, intensity = record_peaks(record)
        fragment_bins = binned_max(mz, intensity)
        neutral_bins = binned_max(float(row['precursor_mz']) - mz, intensity)
        for column, value in fragment_bins.items():
            fragment_rows.append(row_index); fragment_cols.append(column); fragment_data.append(value)
        for column, value in neutral_bins.items():
            neutral_rows.append(row_index); neutral_cols.append(column); neutral_data.append(value)
        ce = pd.to_numeric(pd.Series([row['collision_energy_value']]), errors='coerce').iloc[0]
        ce_missing = not np.isfinite(ce) or ce < 0
        ion_text = str(row['ion_mode']).lower()
        polarity = 1.0 if 'positive' in ion_text else (-1.0 if 'negative' in ion_text else 0.0)
        precursor = float(row['precursor_mz'])
        metadata[row_index] = [precursor if np.isfinite(precursor) else 0.0, polarity, 0.0 if ce_missing else float(ce), float(ce_missing), np.log1p(len(mz)), float(not np.isfinite(precursor))]
        if row_index < 25 and len(mz):
            permutation = np.random.default_rng(SEED + row_index).permutation(len(mz))
            order_invariance_pass &= fragment_bins == binned_max(mz[permutation], intensity[permutation])

X_fragment = sparse.csr_matrix((fragment_data, (fragment_rows, fragment_cols)), shape=(len(spectra), N_BINS), dtype=np.float32)
X_neutral_loss = sparse.csr_matrix((neutral_data, (neutral_rows, neutral_cols)), shape=(len(spectra), N_BINS), dtype=np.float32)
X_presence = X_fragment.copy()
X_presence.data[:] = 1.0
X_presence.eliminate_zeros()

sparse.save_npz(OUTPUT_DIR / 'fragment_intensity_1Da.npz', X_fragment, compressed=True)
sparse.save_npz(OUTPUT_DIR / 'neutral_loss_intensity_1Da.npz', X_neutral_loss, compressed=True)
sparse.save_npz(OUTPUT_DIR / 'fragment_mz_presence_1Da.npz', X_presence, compressed=True)
np.save(OUTPUT_DIR / 'spectrum_metadata.npy', metadata)
spectra[['feature_row', 'spectrum_id', 'inchikey_2d', 'stream_offset', 'cv_fold', 'precursor_mz', 'ion_mode', 'collision_energy_value', 'cleaned_peak_n']].to_csv(OUTPUT_DIR / 'spectrum_feature_row_index.csv', index=False)

print('fragment:', X_fragment.shape, 'nnz:', X_fragment.nnz)
print('neutral loss:', X_neutral_loss.shape, 'nnz:', X_neutral_loss.nnz)
print('峰顺序置换不变性:', order_invariance_pass)

## 3. 化合物等权抽样、固定浅层模型与化合物级聚合

In [ ]:
spectrum_indices_by_key = {key: frame['feature_row'].to_numpy(dtype=int) for key, frame in spectra.groupby('inchikey_2d', sort=True)}
spectrum_id_by_row = spectra.set_index('feature_row')['spectrum_id'].to_dict()

def stable_seed(*parts):
    payload = '|'.join(map(str, parts)).encode('utf-8')
    return int(hashlib.sha256(payload).hexdigest()[:8], 16)

def sample_training_rows(compound_keys, task, outer_fold):
    sampled = []
    for repeat in range(TRAIN_SAMPLING_REPEATS):
        rng = np.random.default_rng(stable_seed(SEED, task, outer_fold, repeat))
        for key in sorted(compound_keys):
            candidates = spectrum_indices_by_key[key]
            sampled.append(int(rng.choice(candidates)))
    return np.asarray(sampled, dtype=int)

def route_matrices(route, train_rows, test_rows):
    if route == 'precursor_mz_only':
        scaler = StandardScaler().fit(metadata[train_rows][:, [0]])
        return scaler.transform(metadata[train_rows][:, [0]]), scaler.transform(metadata[test_rows][:, [0]])
    if route == 'metadata_only':
        scaler = StandardScaler().fit(metadata[train_rows])
        return scaler.transform(metadata[train_rows]), scaler.transform(metadata[test_rows])
    if route == 'fragment_mz_presence':
        return X_presence[train_rows], X_presence[test_rows]
    if route == 'fragment_intensity':
        return X_fragment[train_rows], X_fragment[test_rows]
    if route == MAIN_SPECTRUM_ROUTE:
        scaler = StandardScaler().fit(metadata[train_rows])
        train_meta = sparse.csr_matrix(scaler.transform(metadata[train_rows]).astype(np.float32))
        test_meta = sparse.csr_matrix(scaler.transform(metadata[test_rows]).astype(np.float32))
        return (
            sparse.hstack([X_fragment[train_rows], X_neutral_loss[train_rows], train_meta], format='csr'),
            sparse.hstack([X_fragment[test_rows], X_neutral_loss[test_rows], test_meta], format='csr'),
        )
    raise KeyError(route)

def fit_predict(kind, X_train, y_train, X_test, seed):
    if kind == 'classification':
        estimator = LogisticRegression(C=1.0, solver='liblinear', max_iter=4000, random_state=seed)
        estimator.fit(X_train, np.asarray(y_train, dtype=int))
        probability = estimator.predict_proba(X_test)[:, 1]
        return (probability >= 0.5).astype(float), probability
    estimator = Ridge(alpha=10.0, solver='lsqr', tol=1e-4, max_iter=3000)
    estimator.fit(X_train, np.asarray(y_train, dtype=float))
    prediction = estimator.predict(X_test)
    return np.asarray(prediction, dtype=float), np.full(len(prediction), np.nan)

def aggregate_spectrum_predictions(frame, kind):
    ordered = frame.sort_values(['inchikey_2d', 'spectrum_id'])
    rows = []
    for key, group in ordered.groupby('inchikey_2d', sort=True):
        first = group.iloc[0]
        if kind == 'classification':
            probability = float(group['y_probability'].median())
            single_probability = float(first['y_probability'])
            prediction = float(probability >= 0.5)
            single_prediction = float(single_probability >= 0.5)
        else:
            prediction = float(group['y_prediction'].median())
            single_prediction = float(first['y_prediction'])
            probability = single_probability = np.nan
        rows.append({
            'inchikey_2d': key, 'outer_fold': int(first['outer_fold']), 'y_true': float(first['y_true']),
            'n_spectra': len(group), 'y_prediction': prediction, 'y_probability': probability,
            'single_y_prediction': single_prediction, 'single_y_probability': single_probability,
            'single_spectrum_id': first['spectrum_id'],
        })
    return pd.DataFrame(rows)

## 4. 运行五路线结构簇OOF与20次化合物级标签置换

In [ ]:
spectrum_prediction_frames = []
compound_prediction_frames = []
permutation_prediction_frames = []
total_jobs = len(ROUTES) * len(TASKS) * N_OUTER_FOLDS
progress = tqdm(total=total_jobs, desc='简单谱图基线：路线×任务×外层fold', mininterval=2.0)

for task, task_spec in TASKS.items():
    label_series = pd.to_numeric(modeling[task_spec['column']], errors='coerce')
    label_map = pd.Series(label_series.to_numpy(), index=modeling['inchikey_2d']).to_dict()
    valid_modeling = modeling.loc[label_series.notna(), ['inchikey_2d', 'cv_fold']].copy()
    for outer_fold in range(N_OUTER_FOLDS):
        train_compounds = valid_modeling.loc[valid_modeling['cv_fold'].ne(outer_fold), 'inchikey_2d'].tolist()
        test_compounds = valid_modeling.loc[valid_modeling['cv_fold'].eq(outer_fold), 'inchikey_2d'].tolist()
        train_rows = sample_training_rows(train_compounds, task, outer_fold)
        test_rows = np.concatenate([spectrum_indices_by_key[key] for key in sorted(test_compounds)]).astype(int)
        train_keys = spectra.iloc[train_rows]['inchikey_2d'].to_numpy()
        test_keys = spectra.iloc[test_rows]['inchikey_2d'].to_numpy()
        y_train = np.asarray([label_map[key] for key in train_keys], dtype=float)
        y_test = np.asarray([label_map[key] for key in test_keys], dtype=float)
        for route in ROUTES:
            X_train, X_test = route_matrices(route, train_rows, test_rows)
            prediction, probability = fit_predict(task_spec['kind'], X_train, y_train, X_test, stable_seed(SEED, route, task, outer_fold))
            spectrum_frame = pd.DataFrame({
                'spectrum_id': [spectrum_id_by_row[int(i)] for i in test_rows],
                'inchikey_2d': test_keys, 'representation_route': route, 'task': task,
                'kind': task_spec['kind'], 'outer_fold': outer_fold, 'y_true': y_test,
                'y_prediction': prediction, 'y_probability': probability,
            })
            spectrum_prediction_frames.append(spectrum_frame)
            compound_frame = aggregate_spectrum_predictions(spectrum_frame, task_spec['kind'])
            compound_frame['representation_route'] = route
            compound_frame['task'] = task
            compound_frame['kind'] = task_spec['kind']
            compound_prediction_frames.append(compound_frame)

            if route == MAIN_SPECTRUM_ROUTE:
                unique_train = np.asarray(sorted(train_compounds))
                original_train_labels = np.asarray([label_map[key] for key in unique_train], dtype=float)
                for permutation_index in range(N_LABEL_PERMUTATIONS):
                    rng = np.random.default_rng(stable_seed(SEED, 'permutation', task, outer_fold, permutation_index))
                    shuffled_labels = rng.permutation(original_train_labels)
                    shuffled_map = dict(zip(unique_train, shuffled_labels))
                    permuted_y_train = np.asarray([shuffled_map[key] for key in train_keys], dtype=float)
                    perm_prediction, perm_probability = fit_predict(task_spec['kind'], X_train, permuted_y_train, X_test, stable_seed(SEED, 'permutation_model', task, outer_fold, permutation_index))
                    perm_spectrum = spectrum_frame[['spectrum_id', 'inchikey_2d', 'outer_fold', 'y_true']].copy()
                    perm_spectrum['y_prediction'] = perm_prediction
                    perm_spectrum['y_probability'] = perm_probability
                    perm_compound = aggregate_spectrum_predictions(perm_spectrum, task_spec['kind'])
                    perm_compound['representation_route'] = route
                    perm_compound['task'] = task
                    perm_compound['kind'] = task_spec['kind']
                    perm_compound['permutation_index'] = permutation_index
                    permutation_prediction_frames.append(perm_compound)
            progress.update(1)
progress.close()

spectrum_oof = pd.concat(spectrum_prediction_frames, ignore_index=True)
compound_oof = pd.concat(compound_prediction_frames, ignore_index=True)
permutation_oof = pd.concat(permutation_prediction_frames, ignore_index=True)
print('谱图级OOF行数:', len(spectrum_oof))
print('化合物级OOF行数:', len(compound_oof))
print('置换化合物级OOF行数:', len(permutation_oof))

## 5. 化合物级指标、单谱图对照、置换检验与配对bootstrap

In [ ]:
def classification_metrics(y_true, probability):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)
    prediction = (probability >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        'pr_auc': average_precision_score(y_true, probability), 'roc_auc': roc_auc_score(y_true, probability),
        'mcc': matthews_corrcoef(y_true, prediction), 'balanced_accuracy': balanced_accuracy_score(y_true, prediction),
        'sensitivity': tp / (tp + fn) if tp + fn else np.nan, 'specificity': tn / (tn + fp) if tn + fp else np.nan,
        'brier': brier_score_loss(y_true, probability),
    }

def regression_metrics(y_true, prediction):
    y_true = np.asarray(y_true, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    return {
        'mae': mean_absolute_error(y_true, prediction),
        'rmse': mean_squared_error(y_true, prediction) ** 0.5,
        'r2': r2_score(y_true, prediction),
        'spearman': spearmanr(y_true, prediction).statistic,
    }

metric_rows = []
bootstrap_rng = np.random.default_rng(SEED + 90_000)
for (route, task, kind), frame in compound_oof.groupby(['representation_route', 'task', 'kind'], sort=True):
    for aggregation in ['multi_spectrum_median', 'single_representative']:
        if kind == 'classification':
            probability_column = 'y_probability' if aggregation == 'multi_spectrum_median' else 'single_y_probability'
            result = classification_metrics(frame['y_true'], frame[probability_column])
            primary_metric = 'pr_auc'
        else:
            prediction_column = 'y_prediction' if aggregation == 'multi_spectrum_median' else 'single_y_prediction'
            result = regression_metrics(frame['y_true'], frame[prediction_column])
            primary_metric = 'mae'
        bootstrap_values = []
        for _ in range(N_BOOTSTRAP):
            index = bootstrap_rng.integers(0, len(frame), len(frame))
            sampled = frame.iloc[index]
            if kind == 'classification':
                if sampled['y_true'].nunique() == 2:
                    bootstrap_values.append(average_precision_score(sampled['y_true'], sampled[probability_column]))
            else:
                bootstrap_values.append(mean_absolute_error(sampled['y_true'], sampled[prediction_column]))
        metric_rows.append({
            'representation_route': route, 'task': task, 'kind': kind, 'aggregation': aggregation,
            'n_compounds': len(frame), **result, 'primary_metric': primary_metric,
            'primary_value': result[primary_metric],
            'primary_ci_low': float(np.quantile(bootstrap_values, 0.025)),
            'primary_ci_high': float(np.quantile(bootstrap_values, 0.975)),
        })
metrics = pd.DataFrame(metric_rows)

permutation_metric_rows = []
for (task, kind, permutation_index), frame in permutation_oof.groupby(['task', 'kind', 'permutation_index'], sort=True):
    if kind == 'classification':
        value = average_precision_score(frame['y_true'], frame['y_probability'])
        metric = 'pr_auc'
    else:
        value = mean_absolute_error(frame['y_true'], frame['y_prediction'])
        metric = 'mae'
    permutation_metric_rows.append({'task': task, 'kind': kind, 'permutation_index': permutation_index, 'primary_metric': metric, 'primary_value': value})
permutation_metrics = pd.DataFrame(permutation_metric_rows)
permutation_summary_rows = []
main_metrics = metrics.loc[metrics['representation_route'].eq(MAIN_SPECTRUM_ROUTE) & metrics['aggregation'].eq('multi_spectrum_median')]
for task, null_frame in permutation_metrics.groupby('task'):
    observed = main_metrics.loc[main_metrics['task'].eq(task)].iloc[0]
    null = null_frame['primary_value'].to_numpy(dtype=float)
    if observed['kind'] == 'classification':
        empirical_p = (1 + int((null >= observed['primary_value']).sum())) / (1 + len(null))
    else:
        empirical_p = (1 + int((null <= observed['primary_value']).sum())) / (1 + len(null))
    permutation_summary_rows.append({
        'task': task, 'kind': observed['kind'], 'n_permutations': len(null), 'observed_primary': observed['primary_value'],
        'null_mean': float(null.mean()), 'null_q05': float(np.quantile(null, 0.05)), 'null_q95': float(np.quantile(null, 0.95)), 'empirical_p': empirical_p,
    })
permutation_summary = pd.DataFrame(permutation_summary_rows)

paired_rows = []
paired_rng = np.random.default_rng(SEED + 120_000)
for task, spec in TASKS.items():
    main = compound_oof.loc[compound_oof['representation_route'].eq(MAIN_SPECTRUM_ROUTE) & compound_oof['task'].eq(task)]
    meta = compound_oof.loc[compound_oof['representation_route'].eq('metadata_only') & compound_oof['task'].eq(task)]
    paired = main.merge(meta, on=['inchikey_2d', 'task', 'kind', 'outer_fold', 'y_true'], suffixes=('_spectrum', '_metadata'), validate='one_to_one')
    if spec['kind'] == 'classification':
        observed_delta = average_precision_score(paired['y_true'], paired['y_probability_spectrum']) - average_precision_score(paired['y_true'], paired['y_probability_metadata'])
    else:
        observed_delta = mean_absolute_error(paired['y_true'], paired['y_prediction_metadata']) - mean_absolute_error(paired['y_true'], paired['y_prediction_spectrum'])
    deltas = []
    for _ in range(N_BOOTSTRAP):
        index = paired_rng.integers(0, len(paired), len(paired))
        sample = paired.iloc[index]
        if spec['kind'] == 'classification':
            if sample['y_true'].nunique() < 2:
                continue
            delta = average_precision_score(sample['y_true'], sample['y_probability_spectrum']) - average_precision_score(sample['y_true'], sample['y_probability_metadata'])
        else:
            delta = mean_absolute_error(sample['y_true'], sample['y_prediction_metadata']) - mean_absolute_error(sample['y_true'], sample['y_prediction_spectrum'])
        deltas.append(delta)
    paired_rows.append({
        'task': task, 'kind': spec['kind'], 'n_compounds': len(paired),
        'delta_definition': 'spectrum_PR_minus_metadata_PR' if spec['kind'] == 'classification' else 'metadata_MAE_minus_spectrum_MAE',
        'observed_delta_positive_is_better': observed_delta,
        'ci_low': float(np.quantile(deltas, 0.025)), 'ci_high': float(np.quantile(deltas, 0.975)),
        'bootstrap_p_one_sided_spectrum_not_better': float((1 + np.sum(np.asarray(deltas) <= 0)) / (1 + len(deltas))),
    })
paired_comparison = pd.DataFrame(paired_rows)

display(metrics.loc[metrics['aggregation'].eq('multi_spectrum_median')].sort_values(['task', 'representation_route']))
display(permutation_summary)
display(paired_comparison)

## 6. 结构上限对照、验收和冻结输出

In [ ]:
simple_comparison = metrics.loc[metrics['aggregation'].eq('multi_spectrum_median')].copy()
simple_comparison['comparison_family'] = 'simple_spectrum_baseline'
simple_comparison['model_or_route'] = simple_comparison['representation_route']
structure_comparison = structure_best.copy()
structure_comparison['comparison_family'] = 'structure_upper_bound_nested_selected'
structure_comparison['model_or_route'] = structure_comparison['representation']
comparison_columns = ['comparison_family', 'model_or_route', 'task', 'kind', 'n_compounds', 'primary_metric', 'primary_value', 'primary_ci_low', 'primary_ci_high']
simple_vs_structure = pd.concat([simple_comparison[comparison_columns], structure_comparison[comparison_columns]], ignore_index=True)

checks = []
def add_check(check_id, description, passed, details=''):
    checks.append({'check_id': check_id, 'description': description, 'status': 'PASS' if passed else 'FAIL', 'details': details})

add_check('SPECBASE001', '上游拆分无FAIL', not split_qc['status'].eq('FAIL').any())
add_check('SPECBASE002', '上游谱图审计无FAIL', not spectrum_qc['status'].eq('FAIL').any())
add_check('SPECBASE003', '仅使用20ppm有效Development谱图', len(spectra) == spectrum_audit['counts']['primary_train_20ppm_spectra'], f"{len(spectra)}/{spectrum_audit['counts']['primary_train_20ppm_spectra']}")
add_check('SPECBASE004', '有效Development化合物数一致', len(modeling) == spectrum_audit['counts']['primary_train_20ppm_compounds'], f"{len(modeling)}/{spectrum_audit['counts']['primary_train_20ppm_compounds']}")
add_check('SPECBASE005', '输入谱图均已精确去重且无跨身份碰撞', spectra['exact_dedup_keep'].all() and not spectra['cross_identity_exact_collision'].any())
add_check('SPECBASE006', '稀疏特征行数与谱图数一致', X_fragment.shape[0] == X_neutral_loss.shape[0] == X_presence.shape[0] == len(spectra))
add_check('SPECBASE007', '谱图特征无非有限值', np.isfinite(X_fragment.data).all() and np.isfinite(X_neutral_loss.data).all() and np.isfinite(metadata).all())
add_check('SPECBASE008', '峰顺序置换不改变分箱特征', order_invariance_pass)
expected_coverage = True
for route in ROUTES:
    for task, spec in TASKS.items():
        expected_n = int(pd.to_numeric(modeling[spec['column']], errors='coerce').notna().sum())
        observed = compound_oof.loc[compound_oof['representation_route'].eq(route) & compound_oof['task'].eq(task), 'inchikey_2d']
        expected_coverage &= observed.nunique() == expected_n and len(observed) == expected_n
add_check('SPECBASE009', '每条路线均覆盖各任务全部有效化合物一次', expected_coverage)
add_check('SPECBASE010', '化合物OOF键无重复', not compound_oof.duplicated(['representation_route', 'task', 'inchikey_2d']).any())
add_check('SPECBASE011', '全部五个冻结外层fold均有预测', compound_oof.groupby(['representation_route', 'task'])['outer_fold'].nunique().eq(N_OUTER_FOLDS).all())
add_check('SPECBASE012', '主谱图路线每个任务完成20次标签置换', permutation_oof.groupby('task')['permutation_index'].nunique().eq(N_LABEL_PERMUTATIONS).all())
add_check('SPECBASE013', '未解锁ChemVista外测', not structure_run_manifest['external_test_unlocked'])
add_check('SPECBASE014', '输入路径不含ChemVista文件', not any('chemvista' in str(path).lower() for path in PATHS.values()))
add_check('SPECBASE015', '化合物等权抽样每轮每个训练化合物一张谱图', TRAIN_SAMPLING_REPEATS >= 1)
qc = pd.DataFrame(checks)
assert qc['status'].eq('PASS').all(), qc.loc[qc['status'].eq('FAIL')].to_dict('records')

spectrum_oof.to_csv(OUTPUT_DIR / 'simple_spectrum_spectrum_level_oof.csv.gz', index=False, compression='gzip')
compound_oof.to_pickle(OUTPUT_DIR / 'simple_spectrum_compound_level_oof.pkl')
compound_oof.to_csv(OUTPUT_DIR / 'simple_spectrum_compound_level_oof.csv.gz', index=False, compression='gzip')
permutation_metrics.to_csv(OUTPUT_DIR / 'simple_spectrum_label_permutation_metrics.csv', index=False)
permutation_summary.to_csv(OUTPUT_DIR / 'simple_spectrum_label_permutation_summary.csv', index=False)
metrics.to_csv(OUTPUT_DIR / 'simple_spectrum_metrics.csv', index=False)
paired_comparison.to_csv(OUTPUT_DIR / 'simple_spectrum_vs_metadata_paired_bootstrap.csv', index=False)
simple_vs_structure.to_csv(OUTPUT_DIR / 'simple_spectrum_vs_structure_upper_bound.csv', index=False)
task_counts.to_csv(OUTPUT_DIR / 'simple_spectrum_task_counts.csv', index=False)
qc.to_csv(OUTPUT_DIR / 'simple_spectrum_qc_checks.csv', index=False)

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

run_manifest = {
    'run_version': 'PBT_MSMS_SIMPLE_SPECTRUM_BASELINES_V1',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'random_seed': SEED,
    'external_test_unlocked': False,
    'development_definition': '20ppm eligible frozen Development spectra only',
    'n_spectra': len(spectra), 'n_compounds': len(modeling),
    'routes': ROUTES, 'main_spectrum_route': MAIN_SPECTRUM_ROUTE,
    'feature_definition': {'bin_width_da': BIN_WIDTH_DA, 'max_mz': MAX_MZ, 'n_bins_each': N_BINS, 'intensity': 'within-spectrum normalized sqrt intensity with max aggregation per bin', 'neutral_loss': 'precursor_mz-fragment_mz in [0,2000)'},
    'training': {'compound_equal_weighting': f'one spectrum per compound per repeat, {TRAIN_SAMPLING_REPEATS} repeats', 'classification': 'fixed logistic C=1 liblinear', 'regression': 'fixed ridge alpha=10 lsqr', 'outer_cv': 'frozen Morgan-Butina structure cluster 5-fold'},
    'evaluation': {'primary_unit': 'compound', 'multi_spectrum_aggregation': 'median', 'single_spectrum_sensitivity': True, 'bootstrap_resamples': N_BOOTSTRAP, 'label_permutations': N_LABEL_PERMUTATIONS},
    'tasks': TASKS,
    'input_hashes': {name: sha256_file(path) for name, path in PATHS.items() if name != 'public_stream'},
    'public_stream_sha256_from_passed_upstream_audit': spectrum_audit['public_stream']['sha256'],
    'software': {'python': sys.version, 'executable': sys.executable, 'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn_version, 'platform': platform.platform()},
    'qc_status': 'PASS',
}
with (OUTPUT_DIR / 'simple_spectrum_run_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(run_manifest, handle, ensure_ascii=False, indent=2)
    handle.write('\n')

display(qc)
display(paired_comparison)
print('简单直接谱图基线完成；ChemVista仍未解锁。')

## 7. 结果解释规则

1. 只有化合物级多谱图聚合指标可作为主结果；谱图级行数不能当作样本量。
2. `observed_delta_positive_is_better`及其置信区间用于判断碎片路线是否优于元数据。若区间跨0，应将该端点结论收缩为未证明碎片提供额外信息。
3. 本步骤是Development内部概念验证，不是ChemVista外测，也不是监管PBT确认。
4. 通过本步骤后再训练Set/Spectrum Transformer、DreaMS和SIRIUS概率指纹。